#Q4
First of all, we should divide data frame into 3 clusters and after that we are bound to make 2 linear regressions; one for whole of data and the other for each cluster. in the end, through comparing MSE and R2(R-squared) for both regressions, clustering attitude will be assesed.


In [2]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

Clustering with K=3

In [4]:
df=pd.read_excel('Boston_Housing.xlsx',sheet_name='Data')
df.dropna(inplace=True)

X=df.drop('MEDV',axis=1)
Y=df['MEDV']

scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

kmeans=KMeans(n_clusters=3,random_state=42,n_init=10)
clusters=kmeans.fit_predict(X_scaled)

df['Cluster']=clusters

Training model

In [6]:
X_train_all, X_test_all, Y_train_all, Y_test_all=train_test_split(X, Y, test_size=0.2, random_state=42)

overall_model=LinearRegression()
overall_model.fit(X_train_all, Y_train_all)

cluster_models={}

for cluster_id in sorted(df['Cluster'].unique()):
    cluster_df=df[df['Cluster'] == cluster_id]
    X_cluster=cluster_df.drop(['MEDV', 'Cluster'], axis=1)
    Y_cluster=cluster_df['MEDV']

    X_train_c, X_test_c, Y_train_c, Y_test_c=train_test_split(X_cluster, Y_cluster, test_size=0.2, random_state=42)

    model=LinearRegression()
    model.fit(X_train_c, Y_train_c)
    cluster_models[cluster_id]=model

Results

In [8]:
Y_pred_all=overall_model.predict(X_test_all)
mse_overall=mean_squared_error(Y_test_all, Y_pred_all)
r2_overall=r2_score(Y_test_all, Y_pred_all)

cluster_metrics=[]
for cluster_id, model in cluster_models.items():
    cluster_df=df[df['Cluster']==cluster_id]
    X_cluster=cluster_df.drop(['MEDV', 'Cluster'], axis=1)
    Y_cluster=cluster_df['MEDV']

    X_test_c = X_test_all[df.loc[X_test_all.index, 'Cluster'] == cluster_id]
    Y_test_c = Y_test_all[df.loc[Y_test_all.index, 'Cluster'] == cluster_id]

    Y_pred_c = model.predict(X_test_c)
    mse_c = mean_squared_error(Y_test_c, Y_pred_c)
    r2_c = r2_score(Y_test_c, Y_pred_c)
    cluster_metrics.append({'cluster_id': cluster_id, 'mse': mse_c, 'r2': r2_c})

print("Overall Model(without Clustering):")
print(f"Overall MSE: {mse_overall:.2f}")
print(f"Overall R-squared: {r2_overall:.2f}")
print()

print("Models for Each Cluster:")
total_mse_clusters = 0
for metrics in cluster_metrics:
    print(f"Cluster {metrics['cluster_id']}:")
    print(f"  MSE: {metrics['mse']:.2f}")
    print(f"  R-squared: {metrics['r2']:.2f}")
    total_mse_clusters += metrics['mse']



Overall Model(without Clustering):
Overall MSE: 24.29
Overall R-squared: 0.67

Models for Each Cluster:
Cluster 0:
  MSE: 6.20
  R-squared: 0.76
Cluster 1:
  MSE: 5.87
  R-squared: 0.94
Cluster 2:
  MSE: 26.45
  R-squared: 0.57


Clustering will significanlty make better results rather than experimenting whole data fram